In [2]:
# Smart Task Manager with AI Assistant
# Google Colab-compatible Python conversion of the original JavaFX application.
#
# Run this entire file in one Colab cell, or paste it into a Colab cell.
# It starts a Flask web application and displays it inside the Colab notebook.

import random
import threading
import time
from dataclasses import asdict, dataclass
from typing import Dict, List

from flask import Flask, jsonify, render_template_string, request


# -----------------------------------------------------------------------------
# Application state
# -----------------------------------------------------------------------------

@dataclass
class Task:
    title: str
    description: str
    priority: str


VALID_PRIORITIES = {"High", "Medium", "Low"}
tasks: List[Task] = []
state_lock = threading.RLock()

app = Flask(__name__)


# -----------------------------------------------------------------------------
# Smart assistant logic
# -----------------------------------------------------------------------------

def smart_advice() -> List[str]:
    """Return advice based on the current task list."""
    with state_lock:
        current_tasks = list(tasks)

    if not current_tasks:
        return ["💡 Tip: Start by adding at least one goal today."]

    advice = []
    high_priority = sum(task.priority == "High" for task in current_tasks)

    if high_priority > 3:
        advice.append(
            "⚠️ Warning: Too many high-priority tasks! Try to balance your workload."
        )

    has_exercise = any("exercise" in task.title.lower() for task in current_tasks)
    if not has_exercise:
        advice.append("🏃 Suggestion: Add an exercise-related task for better health.")

    has_break = any("break" in task.title.lower() for task in current_tasks)
    if not has_break:
        advice.append("☕ Suggestion: Don’t forget to take small breaks.")

    motivational_tips = [
        "🌟 Stay consistent with your goals.",
        "🥤 Drink water while working.",
        "📅 Plan your day in advance.",
        "🎯 Focus on one task at a time.",
        "💤 Don’t forget proper rest!",
    ]
    advice.append(random.choice(motivational_tips))
    return advice


def priority_counts() -> Dict[str, int]:
    """Return the number of High, Medium, and Low priority tasks."""
    with state_lock:
        return {
            "High": sum(task.priority == "High" for task in tasks),
            "Medium": sum(task.priority == "Medium" for task in tasks),
            "Low": sum(task.priority == "Low" for task in tasks),
        }


def task_payload(message: str = "") -> Dict:
    """Build the JSON response used by the browser."""
    with state_lock:
        task_data = [asdict(task) for task in tasks]
    return {
        "tasks": task_data,
        "counts": priority_counts(),
        "advice": smart_advice(),
        "message": message,
    }


# -----------------------------------------------------------------------------
# Flask API routes
# -----------------------------------------------------------------------------

@app.get("/")
def home():
    return render_template_string(PAGE)


@app.get("/api/tasks")
def get_tasks():
    return jsonify(task_payload())


@app.post("/api/tasks")
def add_task():
    data = request.get_json(silent=True) or {}
    title = str(data.get("title", "")).strip()
    description = str(data.get("description", "")).strip()
    priority = str(data.get("priority", "")).strip()

    if not title:
        return jsonify(error="Task title is required."), 400
    if len(title) > 120:
        return jsonify(error="Task title must be 120 characters or fewer."), 400
    if len(description) > 500:
        return jsonify(error="Description must be 500 characters or fewer."), 400
    if priority not in VALID_PRIORITIES:
        return jsonify(error="Choose High, Medium, or Low priority."), 400

    with state_lock:
        tasks.append(Task(title=title, description=description, priority=priority))

    return jsonify(task_payload(f"✅ AI: Task added → {title}"))


@app.delete("/api/tasks/<int:task_index>")
def delete_task(task_index: int):
    with state_lock:
        if task_index < 0 or task_index >= len(tasks):
            return jsonify(error="The selected task no longer exists."), 404
        deleted = tasks.pop(task_index)

    return jsonify(task_payload(f"🗑️ AI: Task deleted → {deleted.title}"))


@app.delete("/api/tasks")
def clear_tasks():
    with state_lock:
        tasks.clear()
    return jsonify(task_payload("⚡ AI: All tasks cleared!"))


# -----------------------------------------------------------------------------
# Browser interface
# -----------------------------------------------------------------------------

PAGE = r"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Smart Task Manager</title>
  <style>
    :root {
      font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #172033;
      background: #f3f6fb;
    }
    * { box-sizing: border-box; }
    body { margin: 0; }
    .wrap { max-width: 1100px; margin: 28px auto; padding: 0 18px 30px; }
    .hero { margin-bottom: 18px; }
    .hero h1 { margin: 0 0 6px; font-size: 30px; }
    .hero p { margin: 0; color: #5c6980; }
    .grid { display: grid; grid-template-columns: 1.45fr .85fr; gap: 18px; }
    .card { background: white; border: 1px solid #e1e7f0; border-radius: 16px;
      padding: 18px; box-shadow: 0 8px 24px #182f5510; }
    .toolbar { display: grid; grid-template-columns: 1.1fr 1.5fr .75fr auto; gap: 8px; }
    input, select, button { width: 100%; padding: 11px 12px; border-radius: 9px;
      border: 1px solid #cdd6e3; font: inherit; }
    button { border: 0; font-weight: 650; cursor: pointer; }
    .primary { background: #245fe8; color: white; }
    .danger { background: #fee9e7; color: #ad3026; }
    .muted { background: #edf2f7; color: #233249; }
    .tablebox { overflow: auto; margin-top: 14px; }
    table { width: 100%; border-collapse: collapse; font-size: 14px; }
    th, td { text-align: left; padding: 10px 8px; border-bottom: 1px solid #edf0f5; }
    th { color: #647086; font-size: 12px; text-transform: uppercase; letter-spacing: .04em; }
    .priority { display: inline-block; padding: 4px 8px; border-radius: 999px; font-size: 12px; font-weight: 700; }
    .high { background: #ffe1df; color: #a72b23; }
    .medium { background: #fff1c9; color: #876000; }
    .low { background: #dff6e8; color: #19703b; }
    .small-delete { width: auto; padding: 6px 9px; background: #f5f7fa; color: #455468; }
    .chart-row { display: flex; align-items: center; gap: 10px; margin: 14px 0; }
    .chart-label { width: 70px; font-size: 13px; }
    .bar { flex: 1; height: 22px; background: #edf1f6; border-radius: 99px; overflow: hidden; }
    .fill { height: 100%; border-radius: 99px; transition: width .25s ease; }
    .fill.high { background: #ef6a61; }
    .fill.medium { background: #edbd4b; }
    .fill.low { background: #4dbb76; }
    .count { width: 28px; text-align: right; font-weight: 700; }
    .assistant { min-height: 190px; background: #f8faff; border: 1px solid #e4eafb;
      border-radius: 10px; padding: 12px; }
    .assistant p { margin: 0 0 9px; line-height: 1.4; }
    .status { min-height: 22px; margin-top: 12px; color: #4b5a70; font-size: 13px; }
    .empty { text-align: center; color: #7a8799; padding: 24px 8px; }
    @media (max-width: 800px) {
      .grid { grid-template-columns: 1fr; }
      .toolbar { grid-template-columns: 1fr; }
    }
  </style>
</head>
<body>
  <main class="wrap">
    <header class="hero">
      <h1>🤖 Smart Task Management System</h1>
      <p>Manage your tasks and receive helpful AI-style productivity suggestions.</p>
    </header>

    <div class="grid">
      <section class="card">
        <h2>Tasks</h2>
        <form id="taskForm" class="toolbar">
          <input id="title" maxlength="120" placeholder="Task title" required>
          <input id="description" maxlength="500" placeholder="Description">
          <select id="priority" required>
            <option value="">Priority</option>
            <option>High</option>
            <option>Medium</option>
            <option>Low</option>
          </select>
          <button class="primary" type="submit">Add Task</button>
        </form>
        <div id="status" class="status"></div>
        <div class="tablebox">
          <table>
            <thead><tr><th>Title</th><th>Description</th><th>Priority</th><th></th></tr></thead>
            <tbody id="taskRows"></tbody>
          </table>
        </div>
        <button id="clearButton" class="danger" type="button" style="margin-top:14px;">Clear All Tasks</button>
      </section>

      <aside class="card">
        <h2>📊 Priority Distribution</h2>
        <div id="chart"></div>
        <h2>🤖 AI Assistant</h2>
        <div id="assistant" class="assistant"></div>
      </aside>
    </div>
  </main>

<script>
  const $ = (id) => document.getElementById(id);

  function escapeHtml(value) {
    return String(value).replace(/[&<>'"]/g, (char) => ({
      '&': '&amp;', '<': '&lt;', '>': '&gt;', "'": '&#39;', '"': '&quot;'
    }[char]));
  }

  function render(data) {
    const rows = $('taskRows');
    if (!data.tasks.length) {
      rows.innerHTML = '<tr><td colspan="4" class="empty">No tasks yet. Add your first goal.</td></tr>';
    } else {
      rows.innerHTML = data.tasks.map((task, index) => `
        <tr>
          <td><strong>${escapeHtml(task.title)}</strong></td>
          <td>${escapeHtml(task.description || '—')}</td>
          <td><span class="priority ${task.priority.toLowerCase()}">${task.priority}</span></td>
          <td><button class="small-delete" onclick="deleteTask(${index})">Delete</button></td>
        </tr>`).join('');
    }

    const total = Math.max(1, data.tasks.length);
    $('chart').innerHTML = ['High', 'Medium', 'Low'].map((level) => {
      const count = data.counts[level] || 0;
      const width = Math.round((count / total) * 100);
      return `<div class="chart-row">
        <div class="chart-label">${level}</div>
        <div class="bar"><div class="fill ${level.toLowerCase()}" style="width:${width}%"></div></div>
        <div class="count">${count}</div>
      </div>`;
    }).join('');

    $('assistant').innerHTML = data.advice.map((item) => `<p>${escapeHtml(item)}</p>`).join('');
    if (data.message) $('status').textContent = data.message;
  }

  async function loadTasks() {
    const response = await fetch('/api/tasks');
    render(await response.json());
  }

  $('taskForm').addEventListener('submit', async (event) => {
    event.preventDefault();
    const response = await fetch('/api/tasks', {
      method: 'POST', headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({
        title: $('title').value,
        description: $('description').value,
        priority: $('priority').value
      })
    });
    const data = await response.json();
    if (!response.ok) { $('status').textContent = data.error; return; }
    $('taskForm').reset();
    render(data);
  });

  async function deleteTask(index) {
    const response = await fetch(`/api/tasks/${index}`, {method: 'DELETE'});
    const data = await response.json();
    if (!response.ok) { $('status').textContent = data.error; return; }
    render(data);
  }

  $('clearButton').addEventListener('click', async () => {
    const response = await fetch('/api/tasks', {method: 'DELETE'});
    render(await response.json());
  });

  loadTasks();
</script>
</body>
</html>"""


# -----------------------------------------------------------------------------
# Google Colab launcher
# -----------------------------------------------------------------------------

def start_app(port: int = 5000):
    """Start Flask in the background and display it in Google Colab."""
    server_thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False),
        daemon=True,
    )
    server_thread.start()
    time.sleep(2)

    try:
        from google.colab.output import serve_kernel_port_as_iframe
        print("Smart Task Manager is running below:")
        serve_kernel_port_as_iframe(port, height=760)
    except ImportError:
        print(f"Application is running at http://127.0.0.1:{port}")
        print("In Google Colab, run this code in a Colab cell to display it:")
        print(f"from google.colab.output import serve_kernel_port_as_iframe")
        print(f"serve_kernel_port_as_iframe({port}, height=760)")


if __name__ == "__main__":
    start_app()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Smart Task Manager is running below:


<IPython.core.display.Javascript object>